
# 📘 NLP Pipeline – Arabic PDF Processing

This notebook:
- Extracts text from a PDF
- Cleans Arabic text
- Tokenizes and removes stopwords
- Converts text to TF-IDF

In [1]:
# Installation des prérequis
# Note: pypdf est préféré à PyPDF2 pour les PDFs arabes (meilleur support RTL)
!pip install pypdf scikit-learn pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 7.7 MB/s eta 0:00:00


In [2]:
# Import des bibliothèques
import unicodedata
import re
import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer


In [3]:
# Stopwords arabes complets
ARABIC_STOPWORDS = {
    'من', 'الى', 'إلى', 'عن', 'على', 'في', 'هذا', 'هذه', 'ذلك', 'تلك', 'كان', 'كانت',
    'و', 'ف', 'ثم', 'ل', 'ب', 'ك', 'ما', 'لا', 'لم', 'لن', 'إن', 'إذا', 'عند',
    'بين', 'مع', 'حتى', 'خلال', 'دون', 'بعد', 'قبل', 'أثناء', 'هو', 'هي', 'هم',
    'هن', 'نحن', 'أنت', 'أنتم', 'أنا', 'أن', 'التي', 'الذي', 'الذين', 'أو', 'أي',
    'كل', 'بعض', 'به', 'لها', 'لهم', 'عليها', 'عليهم', 'كما', 'وقد', 'قد',
    'ولقد', 'فقد', 'فإن', 'فهو', 'فهي', 'وكان', 'وكانت', 'الا', 'إلا', 'ولا',
    'وما', 'وأن', 'وبين', 'ففي', 'فمن', 'فكان', 'لكن', 'علي', 'إلي', 'لدى',
    'حين', 'حيث', 'بما', 'بأن', 'لأن', 'لما', 'أين', 'كيف', 'متى', 'لماذا',
    'يمكن', 'يجب', 'لكي', 'لذا', 'ذا', 'لذلك', 'معا', 'معاً', 'اليوم',
}


In [11]:
# ✅ Extraction PDF corrigée pour l'arabe (RTL + formes de présentation)
# 1. Utiliser pypdf avec un "visitor" qui capture les coordonnées (x, y) de chaque token
# 2. Trier par y décroissant (haut → bas) puis x décroissant (droite → gauche pour RTL)
# 3. Normaliser NFKC pour convertir les formes de présentation arabes en caractères de base

def extract_text_from_pdf(pdf_path):
    """
    Extrait le texte arabe d'un PDF en respectant l'ordre RTL.

    Utilise un visitor de position pour reconstruire l'ordre logique des lignes,
    puis applique la normalisation Unicode NFKC pour convertir les formes
    de présentation arabes (ex: ﺗﻘﻨﻴﺎت) en caractères de base (ex: تقنيات).
    """
    r = PdfReader(pdf_path)
    all_text = ''

    for page_num, page in enumerate(r.pages, 1):
        parts = []

        def visitor(text, cm, tm, fd, fs):
            """Capture chaque fragment de texte avec sa position (x, y)."""
            if text.strip():
                # tm[4] = x, tm[5] = y
                parts.append((tm[5], tm[4], text))

        page.extract_text(visitor_text=visitor)

        # Grouper les fragments par ligne (y similaire)
        lines = {}
        for y, x, text in parts:
            y_key = round(y, 0)
            if y_key not in lines:
                lines[y_key] = []
            lines[y_key].append((x, text))

        # Reconstruire le texte: y décroissant (haut→bas), x décroissant (RTL droite→gauche)
        for y_key in sorted(lines.keys(), reverse=True):
            tokens_in_line = sorted(lines[y_key], key=lambda t: -t[0])
            line_text = ' '.join(unicodedata.normalize('NFKC', t) for _, t in tokens_in_line)
            all_text += line_text.strip() + '\n'

    print(f"✅ Extraction réussie: {len(r.pages)} pages, {len(all_text)} caractères")
    return all_text


In [5]:
# Nettoyage du texte arabe
def clean_arabic_text(text):
    """
    Nettoie le texte arabe après extraction correcte.
    Ne conserve que les caractères arabes Unicode (U+0600–U+06FF).
    Après normalisation NFKC, tous les mots sont déjà en forme de base.
    """
    # Supprimer tout ce qui n'est pas arabe ni espace
    text = re.sub(r'[^\u0600-\u06FF\s]', ' ', text)
    # Normaliser les espaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [6]:
# Tokenisation et filtrage
def tokenize_and_filter(text, stopwords_set, min_word_length=3):
    """
    Tokenise et filtre le texte arabe.
    min_word_length=3 recommandé (2 supprimait trop peu de bruit).
    """
    tokens = text.split()
    return [t for t in tokens if len(t) >= min_word_length and t not in stopwords_set]


In [7]:
# Pipeline NLP arabe complet
class ArabicNLPPipeline:

    def __init__(self, pdf_path=None, custom_text=None):
        self.pdf_path = pdf_path
        self.custom_text = custom_text
        self.raw_text = ''
        self.cleaned_text = ''
        self.tokens = []
        self.stopwords = ARABIC_STOPWORDS
        self.tfidf_matrix = None
        self.vectorizer = None

    def load_data(self):
        if self.pdf_path:
            self.raw_text = extract_text_from_pdf(self.pdf_path)
        elif self.custom_text:
            self.raw_text = self.custom_text
        else:
            raise ValueError("Fournir pdf_path ou custom_text")
        return self

    def preprocess(self, min_word_length=3):
        self.cleaned_text = clean_arabic_text(self.raw_text)
        self.tokens = tokenize_and_filter(self.cleaned_text, self.stopwords, min_word_length)
        return self

    def vectorize(self, additional_docs=None):
        corpus = [' '.join(self.tokens)]
        if additional_docs:
            corpus.extend(additional_docs)
        self.vectorizer = TfidfVectorizer(
            token_pattern=r'(?u)\b\w+\b',
            min_df=1,
            max_df=1.0,
            ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(corpus)
        return self

    def get_top_keywords(self, n=20):
        feature_names = self.vectorizer.get_feature_names_out()
        weights = self.tfidf_matrix[0].toarray()[0]
        keywords = [(w, s) for w, s in zip(feature_names, weights) if s > 0]
        return sorted(keywords, key=lambda x: -x[1])[:n]

    def get_statistics(self):
        return {
            'caracteres_bruts': len(self.raw_text),
            'caracteres_nettoyes': len(self.cleaned_text),
            'tokens_filtres': len(self.tokens),
            'vocabulaire_unique': len(set(self.tokens)),
        }

    def run(self, additional_docs=None, min_word_length=3):
        print('='*60)
        print('PIPELINE NLP ARABE - DÉMARRAGE')
        print('='*60)
        print('\n[1/4] Chargement des données...')
        self.load_data()
        print('\n[2/4] Prétraitement...')
        self.preprocess(min_word_length)
        print(f'      {len(self.tokens)} tokens conservés')
        print('\n[3/4] Vectorisation TF-IDF...')
        self.vectorize(additional_docs)
        print('\n[4/4] Analyse terminée!')
        return self


In [8]:
# Affichage des résultats
def display_results(pipeline):
    print('\n' + '='*60)
    print('RÉSULTATS DU PIPELINE')
    print('='*60)

    print('\n--- APERÇU DU TEXTE NETTOYÉ ---')
    preview = pipeline.cleaned_text[:300]
    print(preview + ('...' if len(pipeline.cleaned_text) > 300 else ''))

    print('\n--- PREMIERS TOKENS (15) ---')
    for i, token in enumerate(pipeline.tokens[:15], 1):
        print(f'  {i}. {token}')

    print('\n--- TOP 15 MOTS-CLÉS (TF-IDF) ---')
    for i, (word, score) in enumerate(pipeline.get_top_keywords(15), 1):
        print(f'  {i}. {word}: {score:.4f}')

    stats = pipeline.get_statistics()
    print('\n--- STATISTIQUES ---')
    print(f"  Caractères bruts: {stats['caracteres_bruts']}")
    print(f"  Caractères nettoyés: {stats['caracteres_nettoyes']}")
    print(f"  Tokens filtrés: {stats['tokens_filtres']}")
    print(f"  Vocabulaire unique: {stats['vocabulaire_unique']}")


In [9]:
# Exécution du pipeline
pdf_file = "text.pdf"

additional_docs = [
    "الذكاء الاصطناعي تعلم الآلة معالجة البيانات",
    "تحليل البيانات الضخمة استخراج المعلومات",
    "تطوير أنظمة ذكية للتنبؤ واتخاذ القرارات"
]

pipeline = ArabicNLPPipeline(pdf_path=pdf_file)
pipeline.run(additional_docs=additional_docs, min_word_length=3)
display_results(pipeline)


PIPELINE NLP ARABE - DÉMARRAGE

[1/4] Chargement des données...
✅ Extraction réussie: 1 pages, 1000 caractères

[2/4] Prétraitement...
      124 tokens conservés

[3/4] Vectorisation TF-IDF...

[4/4] Analyse terminée!

RÉSULTATS DU PIPELINE

--- APERÇU DU TEXTE NETTOYÉ ---
من كما في يجب تشير تعتبر تلعب لذلك، تعتمد يمكن تستخدم أهم مجال على هذه لهذه يُنصح تقنيات تستخدم معالجة الدراسات تحديات الطب، الباحثين الذكاء خوارزميات اللغة التقنيات تقنيات الأنظمة الشباب الحديثة الذكاء تساعد تعلم فهم على بتعلم إلى التوصية الطبيعية الآلة والمطورين اللغة الاصطناعي أن تحليل أنظمة في د...

--- PREMIERS TOKENS (15) ---
  1. تشير
  2. تعتبر
  3. تلعب
  4. لذلك،
  5. تعتمد
  6. تستخدم
  7. أهم
  8. مجال
  9. لهذه
  10. يُنصح
  11. تقنيات
  12. تستخدم
  13. معالجة
  14. الدراسات
  15. تحديات

--- TOP 15 MOTS-CLÉS (TF-IDF) ---
  1. مثل: 0.1812
  2. الاصطناعي: 0.1428
  3. الذكاء: 0.1428
  4. أهم: 0.1208
  5. التقنيات: 0.1208
  6. الحديثة: 0.1208
  7. الطب: 0.1208
  8. اللغة: 0.1208
  9. تستخدم: 0.1208
  10. ت

In [10]:
# Export optionnel vers CSV
def export_keywords_to_csv(pipeline, filename="keywords_arabic.csv"):
    keywords = pipeline.get_top_keywords(50)
    df = pd.DataFrame(keywords, columns=["Mot", "Score_TFIDF"])
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n✅ Mots-clés exportés vers {filename}")
